# Part 3 — Clinical Trial Simulation

**Phase A:** Rule set generation — 10 biomedical databases → LLM synthesis → simulation parameters  
**Phase B:** Daily simulation — Hazard functions + LLM enrichment → synthetic clinical trial data

---

This notebook is part of the **MedGemma Clinical Trial Engine** pipeline:

```
NB1  Anti-Hallucination ──── RLFR fine-tuning for reliable medical text (MedGemma)
NB2  SAE Detection ────────── MedGemma 1.5 + MedSigLIP + HeAR (image + audio → AE)
NB3  Clinical Trial Sim ──── Rule set generation + hazard-based daily simulation
NB4  Voice Call App ────────── MedGemma 4B virtual nurse (multi-turn dialogue)
NB5  SAE Report Gen ────────── CRF data → MedWatch 3500A pharmacovigilance reports
```

---

# Phase A — Rule Set Generation

Generate simulation parameters from 10 biomedical databases + LLM synthesis.

---
## 0. Environment Setup

Before running the pipeline, you need:
1. **The project files** — the notebook auto-detects the project root (checks current directory, `./ruleset_generation/`, and Kaggle input paths)
2. **Gemini API key** — free at https://aistudio.google.com/apikey (store as Kaggle Secret `GEMINI_API_KEY` or set `RULE_ENGINE_LLM_API_KEY` env var)
3. (Optional) Run `setup.sh` to download local data files for improved accuracy

In [ ]:
import os, sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore", message=".*pickleshare.*")

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

# For Phase A: rule_engine imports
RULESET_DIR = ROOT / 'src' / 'ruleset_generation'
sys.path.insert(0, str(RULESET_DIR))
sys.path.insert(0, str(ROOT))

os.chdir(RULESET_DIR)
print(f"Project root: {ROOT}")
print(f"Rule engine:  {RULESET_DIR}")

In [ ]:
# Install dependencies (run once)
!pip install -q google-genai openai pydantic pydantic-settings httpx requests aiohttp numpy pandas matplotlib seaborn nest-asyncio jsonschema scipy typer rich

# Install this project (for src.* imports)
!pip install -q -e ..

In [ ]:
import sys, os, json, time, asyncio
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from IPython.display import display, HTML, Markdown

import nest_asyncio
nest_asyncio.apply()

PROJECT_ROOT = ROOT

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100

def run_async(coro):
    """Run an async coroutine from sync context."""
    try:
        loop = asyncio.get_running_loop()
        return loop.run_until_complete(coro)
    except RuntimeError:
        return asyncio.run(coro)

print(f"Project root: {PROJECT_ROOT}")

### Download Data Files (optional — skip if you want)

The pipeline can run with just the 6 **API sources** (DailyMed, CT.gov, OpenFDA, PubChem, ChEMBL, PubMed) and no local data. However, downloading the 3 local databases improves accuracy by adding OnSIDES ADE validation, PrimeKG knowledge graph context, and DrugBank pharmacology data.

Run `setup.sh` to download and process them automatically:

| Dataset | Size | What It Provides |
|---------|------|------------------|
| **PrimeKG** | ~925 MB | Knowledge graph — drug-disease, drug-gene relationships |
| **DrugBank** | ~170 MB | Drug-target binding, pharmacological properties |
| **OnSIDES** | ~2 GB | 7.1M validated drug-ADE pairs from 51,460 FDA labels |

**Skip this cell** if you don't need local data or have already run `setup.sh`.

In [ ]:
# Download local databases (PrimeKG, DrugBank, OnSIDES) — skips if already present
# These 3 databases add ~3.5 GB of evidence data that dramatically improves results
!bash setup.sh 2>&1 | grep -E '^\[|^=|Done|passed|Complete|Next|Skipped|already|error|Error' | head -30

### Gemini API Key (Required)

The pipeline uses **Gemini 2.0 Flash** via OpenAI-compatible endpoint.  
Get a free API key at https://aistudio.google.com/apikey

**On Kaggle:** Store your key as a Kaggle Secret named `GEMINI_API_KEY`.  
**Locally:** Set the `RULE_ENGINE_LLM_API_KEY` environment variable, or you will be prompted to enter it.

In [ ]:
import os
from pathlib import Path

# ── Google API key (Gemini) ──
# Get a free key at: https://aistudio.google.com/apikey
GOOGLE_API_KEY = ""  # <-- paste your Gemini API key here

if not GOOGLE_API_KEY:
    GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY", "")

assert GOOGLE_API_KEY, (
    "Gemini API key not found. Either:\n"
    "  1. Set GOOGLE_API_KEY = 'your-key' in this cell\n"
    "  2. Set GOOGLE_API_KEY environment variable\n"
    "  3. Get a free key at https://aistudio.google.com/apikey"
)
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
os.environ["RULE_ENGINE_LLM_API_KEY"] = GOOGLE_API_KEY
print(f"Gemini API key: ...{GOOGLE_API_KEY[-6:]}")

### Define Drug + Indication

We'll use **Paclitaxel + Carboplatin** for Non-Small Cell Lung Cancer as the example — both drugs have structured DailyMed AE tables and rich evidence across all 10 sources. You can change `DRUGS` and `INDICATION` below to generate rulesets for any drug-indication pair.

In [ ]:
# Change these to generate rulesets for different drugs
DRUGS = ["Paclitaxel", "Carboplatin"]
INDICATION = "Non-Small Cell Lung Cancer"

DRUG_LABEL = "+".join(DRUGS).lower()
INDICATION_LABEL = INDICATION.lower().replace(" ", "_")
OUTPUT_SUBDIR = f"{DRUG_LABEL}_{INDICATION_LABEL}"

print(f"Drug(s):     {' + '.join(DRUGS)}")
print(f"Indication:  {INDICATION}")
print(f"Output dir:  output/{OUTPUT_SUBDIR}/")

In [ ]:
from rule_engine.config import RuleEngineConfig

config = RuleEngineConfig()

print(f"LLM Model:    {config.llm_model}")
print(f"Rate Limit:   {config.rate_limit_rpm} RPM")
print(f"OnSIDES DB:   {'exists' if config.onsides_db.exists() else 'MISSING'}")
print(f"PrimeKG:      {'exists' if config.primekg_nodes.exists() else 'MISSING'}")
print(f"PDS data:     {'exists' if config.pds_data_dir.exists() else 'not downloaded'}")

---
## 1. Evidence Collection

The pipeline queries **10 databases** in parallel, organized into 3 categories:

| Category | Sources | Access |
|----------|---------|--------|
| **API** (6) | DailyMed, ClinicalTrials.gov, OpenFDA, PubChem, ChEMBL, PubMed | Public, no credentials |
| **Local DB** (3) | PrimeKG, DrugBank, OnSIDES | Downloaded by `setup.sh` |
| **Remote** (1) | Project Data Sphere | PDS account required |

### 1a. API Sources (6 databases, no credentials needed)

These query public REST APIs in real time.

In [ ]:
from rule_engine.evidence.dailymed import fetch_dailymed
from rule_engine.evidence.clinical_trials import fetch_clinical_trials, fetch_clinical_trials_combo
from rule_engine.evidence.openfda import fetch_openfda_aes
from rule_engine.evidence.pubchem import fetch_pubchem
from rule_engine.evidence.chembl import fetch_chembl
from rule_engine.evidence.literature import fetch_literature

print("All API evidence modules loaded.")

#### DailyMed — FDA-Approved Drug Labels

DailyMed provides the **official FDA label** for each drug, including dosage, boxed warnings, and (when available) structured AE incidence tables with grade 3-4 breakdowns.

**Note:** Older drugs like Cisplatin and Etoposide have their adverse reactions as narrative text rather than structured tables. When DailyMed lacks a structured AE table, the pipeline falls back to **ClinicalTrials.gov reported AEs** for frequency data.

In [ ]:
# DailyMed — FDA-approved drug labels (AE incidence tables, dosage, boxed warnings)
dm_results = {}
for drug in DRUGS:
    dm = await fetch_dailymed(drug)
    dm_results[drug] = dm
    boxed = ", has boxed warning" if dm.boxed_warning else ""
    if not dm.found:
        print(f"  DailyMed [{drug}]: NOT FOUND")
    elif dm.ae_table:
        print(f"  DailyMed [{drug}]: {len(dm.ae_table)} AEs in structured table{boxed}")
    else:
        print(f"  DailyMed [{drug}]: Label found (narrative format, no structured AE table){boxed}")
        print(f"    → AE frequencies will come from ClinicalTrials.gov instead")

# Show sample AE table if available
for drug, dm in dm_results.items():
    if dm.ae_table:
        print(f"\n  Sample AEs from {drug} DailyMed label:")
        for ae in dm.ae_table[:5]:
            print(f"    {ae['term']:30s}  {ae['incidence_pct']:5.1f}%  (grade 3-4: {ae.get('grade34_pct', 'N/A')}%)")

#### ClinicalTrials.gov — Trial Demographics & Endpoints

ClinicalTrials.gov provides trial-level data: how many trials exist, what phase they reached, baseline demographics (age, sex), and reported AEs. For combination therapy, a dedicated combo search finds trials that used the exact drug combination.

In [ ]:
# ClinicalTrials.gov — trial demographics, endpoints, reported AEs
ct_results = {}
for drug in DRUGS:
    ct = await fetch_clinical_trials(drug, INDICATION)
    ct_results[drug] = ct
    print(f"  CT.gov [{drug}]: {ct.trial_count} trials, max phase {ct.max_phase}, "
          f"{len(ct.reported_aes)} reported AEs")

# Combo trial search (for combination therapy)
if len(DRUGS) > 1:
    combo_ct = await fetch_clinical_trials_combo(DRUGS, INDICATION)
    print(f"  CT.gov [combo]: {combo_ct.trial_count} trials")
    if combo_ct.baseline_demographics:
        bd = combo_ct.baseline_demographics
        print(f"    Demographics: age={bd.get('age_mean', '?')} +/- {bd.get('age_std', '?')}, "
              f"male={bd.get('sex_male', '?')}, female={bd.get('sex_female', '?')}")

#### OpenFDA/FAERS — Post-Market Adverse Event Reports

The FDA Adverse Event Reporting System (FAERS) contains real-world AE reports submitted after drug approval. Useful for rare AEs not captured in clinical trials and for time-to-onset data.

In [ ]:
# OpenFDA/FAERS — post-market adverse event reports
fda_results = {}
for drug in DRUGS:
    fda = await fetch_openfda_aes(drug)
    fda_results[drug] = fda
    print(f"  OpenFDA [{drug}]: {fda.total_ae_reports} reports, "
          f"{len(fda.top_adverse_events)} top AEs, "
          f"timing data: {'Yes' if fda.has_timing_data else 'No'}")

#### PubChem, ChEMBL & PubMed — Molecular Properties, Bioactivity & Literature

These three sources provide complementary chemical and literature context:
- **PubChem**: Molecular weight, logP, Lipinski violations (drug-likeness)
- **ChEMBL**: Mechanism of action, bioactivity data, max clinical phase
- **PubMed**: Literature co-occurrence score (how often drug + indication appear together)

In [ ]:
# PubChem, ChEMBL, PubMed — molecular properties, bioactivity, literature
for drug in DRUGS:
    pc = await fetch_pubchem(drug)
    ch = await fetch_chembl(drug)
    lit = await fetch_literature(drug, INDICATION)
    print(f"  {drug}:")
    print(f"    PubChem: MW={pc.molecular_weight}, logP={pc.logp}, Lipinski violations={pc.lipinski_violations}")
    print(f"    ChEMBL:  MoA='{ch.mechanism_of_action or 'N/A'}', phase={ch.max_phase}")
    print(f"    PubMed:  {lit.article_count} articles, co-occurrence score={lit.cooccurrence_score:.2f}")

#### Visualize: AE Frequencies per Drug

The chart below shows the **top 15 AEs by incidence** from each drug's FDA label (DailyMed). For older drugs like Cisplatin and Etoposide whose labels use narrative text instead of structured tables, the pipeline falls back to **ClinicalTrials.gov reported AEs**.

**What to look for:**
- **Red bars (>30%)**: Very common AEs — these should definitely appear in the final ruleset
- **Orange bars (10-30%)**: Common AEs — most will be retained after pruning
- **Blue bars (<10%)**: Less common — may be pruned if not corroborated by other sources

In [ ]:
# Plot AE frequencies for each drug
# Uses DailyMed AE table when available; falls back to ClinicalTrials.gov reported AEs
fig, axes = plt.subplots(1, len(DRUGS), figsize=(7 * len(DRUGS), 6), squeeze=False)

for idx, (drug, dm) in enumerate(dm_results.items()):
    ax = axes[0][idx]
    if dm.ae_table:
        # DailyMed structured table available
        top = sorted(dm.ae_table, key=lambda x: x["incidence_pct"], reverse=True)[:15]
        terms = [ae["term"][:25] for ae in top]
        freqs = [ae["incidence_pct"] for ae in top]
        source_label = "DailyMed AE Table"
    elif drug in ct_results and ct_results[drug].reported_aes:
        # Fallback: use ClinicalTrials.gov reported AEs
        ct_aes = ct_results[drug].reported_aes
        # CT.gov reported_aes are dicts with term + stats
        scored = []
        for ae in ct_aes:
            if isinstance(ae, dict) and ae.get("pct") is not None:
                scored.append(ae)
            elif isinstance(ae, dict) and ae.get("count") is not None and ae.get("at_risk"):
                ae["pct"] = 100.0 * ae["count"] / ae["at_risk"]
                scored.append(ae)
        scored = sorted(scored, key=lambda x: x.get("pct", 0), reverse=True)[:15]
        terms = [ae.get("term", ae.get("ae_term", "?"))[:25] for ae in scored]
        freqs = [ae.get("pct", 0) for ae in scored]
        source_label = "ClinicalTrials.gov Reported AEs"
    else:
        ax.text(0.5, 0.5, "No AE frequency data", ha="center", va="center", fontsize=14)
        ax.set_title(drug)
        continue

    colors = ["#e74c3c" if f > 30 else "#f39c12" if f > 10 else "#3498db" for f in freqs]
    ax.barh(terms[::-1], freqs[::-1], color=colors[::-1])
    ax.set_xlabel("Incidence (%)")
    ax.set_title(f"{drug} — {source_label} (top 15)")

plt.tight_layout()
plt.show()

### 1b. Local Database Sources (3 databases)

These read from files downloaded by `setup.sh`. No internet needed.

In [ ]:
from rule_engine.evidence.local_dbs import fetch_drugbank, fetch_primekg
from rule_engine.evidence.onsides import fetch_onsides

for drug in DRUGS:
    # DrugBank — drug-target binding, pharmacology
    db = await fetch_drugbank(drug, config)
    print(f"  DrugBank [{drug}]: found={db.found}, targets={len(db.targets)}, DDIs={db.ddi_count}")
    if db.moa:
        print(f"    MoA: {db.moa[:100]}...")

    # PrimeKG — knowledge graph (drug-disease, drug-gene)
    kg = await fetch_primekg(drug, INDICATION, config)
    print(f"  PrimeKG  [{drug}]: found={kg.found}, diseases={len(kg.disease_associations)}, "
          f"genes={len(kg.gene_targets)}")

    # OnSIDES — 7.1M validated drug-ADE pairs from FDA labels
    ons = await fetch_onsides(drug, config)
    print(f"  OnSIDES  [{drug}]: found={ons.found}, {ons.total_pairs} ADE pairs, "
          f"boxed warnings={len(ons.boxed_warning_aes)}")
    if ons.boxed_warning_aes:
        print(f"    Boxed warnings: {', '.join(ons.boxed_warning_aes[:5])}")
    print()

#### Visualize: OnSIDES ADE Pair Prediction Scores

OnSIDES uses a PubMedBERT model to extract drug-ADE pairs from FDA labels with confidence scores. Higher prediction scores indicate stronger evidence for the association.

**What to look for:**
- **Red bars**: Boxed warning AEs — these are injected into the final ruleset if missing
- **Score > 0.8**: High-confidence associations that corroborate DailyMed data
- Compare with DailyMed: AEs appearing in both sources have stronger evidence

In [ ]:
# OnSIDES top ADE pairs by prediction score
fig, axes = plt.subplots(1, len(DRUGS), figsize=(7 * len(DRUGS), 6), squeeze=False)

for idx, drug in enumerate(DRUGS):
    ax = axes[0][idx]
    ons = await fetch_onsides(drug, config)
    if ons.ae_pairs:
        # Filter out pairs with None scores before sorting
        scored_pairs = [p for p in ons.ae_pairs if p.get("mean_pred_score") is not None]
        top = sorted(scored_pairs, key=lambda x: x["mean_pred_score"], reverse=True)[:15]
        terms = [p["pt_meddra_term"][:25] for p in top]
        scores = [p["mean_pred_score"] for p in top]
        colors = ["#e74c3c" if p.get("is_boxed_warning") else "#3498db" for p in top]
        ax.barh(terms[::-1], scores[::-1], color=colors[::-1])
        ax.set_xlabel("Prediction Score")
        ax.set_title(f"{drug} — OnSIDES Top ADE Pairs")
        # Legend
        from matplotlib.patches import Patch
        ax.legend(handles=[Patch(color="#e74c3c", label="Boxed Warning"),
                           Patch(color="#3498db", label="Standard")], loc="lower right")
    else:
        ax.text(0.5, 0.5, "No OnSIDES data", ha="center", va="center", fontsize=14)
        ax.set_title(drug)

plt.tight_layout()
plt.show()

### 1c. Remote Source — Project Data Sphere (optional)

PDS provides **patient-level** data from real clinical trials.  
Requires a free account at https://projectdatasphere.org

If PDS data is cached locally (from a previous `setup.sh` run), it will be used even without live credentials.

In [ ]:
from rule_engine.evidence.projectdatasphere import fetch_pds

pds = await fetch_pds(DRUGS, INDICATION, config)

if pds.found:
    t = pds.matched_trial
    d = pds.demographics
    print(f"  PDS match: trial={t.trial_id}, n={t.n_patients}, score={t.match_score:.2f}")
    print(f"  Demographics: n={d.n_patients}, age={d.age_mean:.1f} +/- {d.age_std:.1f}, "
          f"male={d.pct_male:.1f}%, female={d.pct_female:.1f}%")
    print(f"  AEs: {len(pds.ae_aggregates)} unique terms")
    print(f"  Efficacy: ORR={pds.efficacy.overall_response_rate_pct}%")
    print(f"  Regimen: {len(pds.regimen)} drugs")
    for r in pds.regimen:
        print(f"    {r.drug}: {r.median_dose} {r.dose_unit} ({r.route})")
else:
    print("  PDS: No matching trial found (or data not downloaded).")
    print("  The pipeline will use the other 9 sources.")

#### Visualize: PDS Demographics (if available)

If PDS matched a trial, these charts show the **real patient-level distributions** — far more accurate than aggregate summaries from ClinicalTrials.gov. PDS demographics always override CT.gov when available.

**What to look for:**
- **Age**: Is the distribution skewed? Oncology trials typically center around 60-65 years
- **Sex**: Many lung cancer trials skew male (60-70%)
- **ECOG**: Most trials enroll ECOG 0-1; ECOG 2+ patients are often excluded

In [ ]:
if pds.found and pds.demographics.n_patients > 0:
    d = pds.demographics
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # Age distribution (simulated from mean/std)
    ax = axes[0]
    ages = np.random.normal(d.age_mean, d.age_std, 1000)
    ages = ages[(ages >= 18) & (ages <= 95)]
    ax.hist(ages, bins=25, color="#3498db", edgecolor="white")
    ax.axvline(d.age_mean, color="red", linestyle="--", label=f"Mean={d.age_mean:.1f}")
    ax.set_xlabel("Age")
    ax.set_title(f"PDS Age Distribution (n={d.n_patients})")
    ax.legend()

    # Sex
    ax = axes[1]
    ax.pie([d.pct_male, d.pct_female], labels=["Male", "Female"],
           autopct="%1.1f%%", colors=["#3498db", "#e74c3c"], startangle=90)
    ax.set_title("Sex Distribution")

    # ECOG
    ax = axes[2]
    if d.ecog_distribution:
        ecog_labels = [f"ECOG {k}" for k in sorted(d.ecog_distribution.keys())]
        ecog_vals = [d.ecog_distribution[k] for k in sorted(d.ecog_distribution.keys())]
        ax.bar(ecog_labels, ecog_vals, color="#2ecc71")
        ax.set_ylabel("Proportion")
        ax.set_title("ECOG Performance Status")
    else:
        ax.text(0.5, 0.5, "No ECOG data", ha="center", va="center")

    plt.tight_layout()
    plt.show()
else:
    print("PDS data not available — skipping visualization.")

### Evidence Summary

In [ ]:
# Collect all evidence via the orchestrator (same as pipeline does internally)
from rule_engine.evidence.collector import collect_evidence
import logging

# Suppress API retry noise (PubMed/ChEMBL rate-limits on repeated runs)
for logger_name in ["rule_engine", "httpx", "httpcore"]:
    logging.getLogger(logger_name).setLevel(logging.ERROR)

print("Collecting all evidence via orchestrator (parallel)...")
bundle = await collect_evidence(DRUGS, INDICATION, config)
print("Done!\n")

# Restore logging
logging.getLogger("rule_engine").setLevel(logging.WARNING)

# Summary table
rows = []
for drug in DRUGS:
    d = bundle.per_drug.get(drug)
    if d:
        dm_count = len(d.dailymed.ae_table) if d.dailymed.found else 0
        dm_label = f"{dm_count:3d} AEs" if dm_count > 0 else "narrative"
        rows.append({
            "Drug": drug,
            "DailyMed": dm_label,
            "OpenFDA Reports": d.openfda.total_ae_reports,
            "OnSIDES Pairs": d.onsides.total_pairs,
            "ChEMBL Activities": d.chembl.activity_count,
            "DrugBank Targets": len(d.drugbank.targets),
        })

ct = bundle.clinical_trials
print("Per-Drug Evidence:")
for r in rows:
    print(f"  {r['Drug']:20s}  DailyMed={r['DailyMed']:>9s}  OpenFDA={r['OpenFDA Reports']:6d} reports  "
          f"OnSIDES={r['OnSIDES Pairs']:4d} pairs  ChEMBL={r['ChEMBL Activities']:3d}  DrugBank={r['DrugBank Targets']:2d} targets")

print(f"\nShared Evidence:")
print(f"  ClinicalTrials.gov: {ct.trial_count} trials, {len(ct.reported_aes)} reported AEs")
print(f"  Combo trials:       {bundle.combo_trials.trial_count} trials")
print(f"  PrimeKG:            {bundle.primekg.found}")
print(f"  PDS:                {'matched' if bundle.pds.found else 'no match'}")

---
## 2. Run the Full Pipeline

Now we run the complete multi-stage LLM pipeline:  
**Stage 1** (5 parallel extractions) → **Stage 2** (grounding) → **Stage 3** (synthesis) → **Overrides** → **Validation** → **Output**

This takes ~2-3 minutes per attempt and makes 7-12 LLM calls. The pipeline retries up to 3 times if Gemini produces unparseable JSON (non-deterministic failure).

**What to expect:**
- Progress messages as each stage completes
- Stage 2 may report a JSON parse failure — this is normal (programmatic overrides compensate)
- The pipeline auto-corrects hallucination patterns and writes split output (`base.json` + type overlay)
- If all 3 attempts fail, the notebook falls back to pre-generated output for visualization

In [ ]:
from rule_engine.pipeline import run_pipeline
import time, logging

# Enable multi-stage mode (3-stage LLM pipeline with programmatic overrides)
config.multi_stage = True

# Suppress noisy stderr from API retries during pipeline execution
logging.getLogger("rule_engine").setLevel(logging.ERROR)
logging.getLogger("httpx").setLevel(logging.ERROR)

print(f"Running pipeline for {' + '.join(DRUGS)} / {INDICATION}...")
print(f"This will make 7-12 LLM calls to {config.llm_model}.\n")

# Retry up to 3 times — Gemini occasionally produces unparseable JSON
MAX_RETRIES = 3
start = time.time()

for attempt in range(1, MAX_RETRIES + 1):
    if attempt > 1:
        print(f"\n--- Retry {attempt}/{MAX_RETRIES} (LLM JSON parse failures are non-deterministic) ---\n")
    result = await run_pipeline([(DRUGS, INDICATION)], config)
    if result.successful:
        break

elapsed = time.time() - start
logging.getLogger("rule_engine").setLevel(logging.WARNING)

print(f"\nPipeline completed in {elapsed:.1f}s ({attempt} attempt{'s' if attempt > 1 else ''})")
print(f"  Successful: {len(result.successful)}")
print(f"  Failed:     {len(result.failed)}")

if result.successful:
    drugs, indication, output_path = result.successful[0]
    drugs_key = "+".join(drugs)
    print(f"  Output:     {output_path.name}/")
    warnings_list = result.warnings.get(drugs_key, [])
    print(f"  Warnings:   {len(warnings_list)}")
elif result.failed:
    print(f"\nPipeline failed after {MAX_RETRIES} attempts. Using pre-generated output for visualization.")

---
## 3. Inspect the Agent Log (Stage by Stage)

The pipeline records every step in an agent log JSON file. This is invaluable for debugging — it shows exactly which LLM calls were made, what evidence was used, which overrides fired, and what corrections the validator applied.

**What to look for:**
- Which Stage 1 sub-calls succeeded vs. failed (severity extraction typically fails)
- How many AE frequency corrections were applied (evidence vs. LLM values)
- How many AEs were pruned for lacking evidence
- Which LLM re-prompts were triggered (indicates missing data)

In [ ]:
# Load agent log (pipeline writes to rule_sets/, pre-generated output is in output/)
log_path = None
for search_dir in [config.output_dir, PROJECT_ROOT / "output"]:
    candidate = search_dir / f"{OUTPUT_SUBDIR}_agent_log.json"
    if candidate.exists():
        log_path = candidate
        break
    # Fallback: glob for drug name
    candidates = list(search_dir.glob(f"*{DRUGS[0].lower()}*agent_log.json"))
    if candidates:
        log_path = sorted(candidates, key=lambda p: p.stat().st_mtime, reverse=True)[0]
        break

if log_path is None:
    print("Agent log not found.")
    agent_log = None
else:
    agent_log = json.loads(log_path.read_text())

    print(f"Agent log: {log_path.name}")
    print(f"Model:     {agent_log['model']}")
    print(f"Timestamp: {agent_log['timestamp']}")
    print(f"Warnings:  {len(agent_log['warnings'])}")
    print(f"Stages:    {len(agent_log['stage_logs'])}")
    print()

    # Show all stages
    print("Pipeline stages executed:")
    print("-" * 60)
    for i, sl in enumerate(agent_log["stage_logs"]):
        stage = sl.get("stage", "unknown")
        if "error" in sl:
            print(f"  {i:2d}. {stage:40s}  ERROR")
        elif "prompt_len" in sl:
            print(f"  {i:2d}. {stage:40s}  LLM call (prompt={sl['prompt_len']:,} chars, response={sl['response_len']:,} chars)")
        elif "corrections" in sl:
            print(f"  {i:2d}. {stage:40s}  {len(sl['corrections'])} corrections")
        elif "pruned" in sl:
            print(f"  {i:2d}. {stage:40s}  pruned={sl['pruned']}, merged={sl['merged']}, remaining={sl['remaining']}")
        else:
            print(f"  {i:2d}. {stage:40s}  {list(sl.keys())}")

### Stage 1 Results: LLM Extraction

In [ ]:
# Stage 1 sub-calls
if agent_log:
    stage1_names = ["stage1_ae_freq", "stage1_severity", "stage1_onset", "stage1_triggers", "stage1_demographics"]
    stage1_logs = [sl for sl in agent_log["stage_logs"] if sl.get("stage") in stage1_names]

    if stage1_logs:
        print("Stage 1 — Parallel LLM Extraction (5 calls):")
        print("-" * 60)
        for sl in stage1_logs:
            name = sl["stage"].replace("stage1_", "")
            print(f"  {name:15s}  prompt={sl.get('prompt_len', 0):>6,} chars  "
                  f"response={sl.get('response_len', 0):>6,} chars")
    else:
        print("Stage 1 — Single-shot mode (no per-stage breakdown)")
        print(f"  Evidence prompt: {len(agent_log.get('evidence_prompt', '')):,} chars")
        print(f"  LLM response:    {len(agent_log.get('raw_response', '')):,} chars")
else:
    print("Agent log not available — skipping Stage 1 details.")

### Programmatic Overrides

In [ ]:
if agent_log:
    printed_something = False

    # Dose override
    dose_log = next((sl for sl in agent_log["stage_logs"] if sl.get("stage") == "programmatic_dose_override"), None)
    if dose_log and "structured_doses" in dose_log:
        print("Dose Override (evidence-based):")
        for drug, info in dose_log["structured_doses"].items():
            print(f"  {drug}: {info.get('dose_value', 'N/A')} ({info.get('dose_unit', '')}) via {info.get('route', '')}")
        printed_something = True

    # AE frequency corrections
    ae_freq_log = next((sl for sl in agent_log["stage_logs"] if sl.get("stage") == "ae_frequency_correction"), None)
    if ae_freq_log and "corrections" in ae_freq_log:
        corrections = ae_freq_log["corrections"]
        print(f"\nAE Frequency Corrections: {len(corrections)} AEs adjusted")
        for c in corrections[:10]:
            print(f"  {c}")
        if len(corrections) > 10:
            print(f"  ... and {len(corrections) - 10} more")
        printed_something = True

    # AE pruning
    prune_log = next((sl for sl in agent_log["stage_logs"] if sl.get("stage") == "ae_evidence_pruning"), None)
    if prune_log:
        print(f"\nAE Pruning: {prune_log.get('pruned', 0)} removed, "
              f"{prune_log.get('merged', 0)} merged, {prune_log.get('remaining', 0)} remaining")
        printed_something = True

    if not printed_something:
        # Summarize what the agent log does contain
        n_stages = len(agent_log["stage_logs"])
        n_warnings = len(agent_log.get("warnings", []))
        print(f"Programmatic overrides: {n_stages} stages logged, {n_warnings} validation warnings")
        if n_stages == 0:
            print("  (Pipeline ran in single-shot mode — overrides are applied inline)")
        # Show any stage names that do exist
        for sl in agent_log["stage_logs"]:
            stage = sl.get("stage", "unknown")
            print(f"  Stage: {stage}")
else:
    print("Agent log not available — skipping override details.")

### Validation Warnings

In [ ]:
if agent_log:
    warnings = agent_log.get("warnings", [])
    print(f"Validation produced {len(warnings)} warnings:\n")

    # Group by type
    from collections import Counter
    prefixes = [w.split(":")[0] if ":" in w else w[:40] for w in warnings]
    for prefix, count in Counter(prefixes).most_common():
        print(f"  {count:2d}x  {prefix}")
else:
    print("Agent log not available — skipping validation warnings.")

---
## 4. Visualize the Final Output

Load the generated `base.json` and visualize key sections.

In [ ]:
# Load output (check both rule_sets/ and output/ directories)
output_dir = config.output_dir / OUTPUT_SUBDIR
if not output_dir.exists():
    output_dir = PROJECT_ROOT / "output" / OUTPUT_SUBDIR

base = json.loads((output_dir / "base.json").read_text())

# Detect schema type from overlay file
overlay_files = [f for f in output_dir.iterdir() if f.name != "base.json" and f.suffix == ".json" and "agent_log" not in f.name]
schema_type = overlay_files[0].stem if overlay_files else "unknown"

print(f"Drug:        {base['drug_name']}")
print(f"Indication:  {base['indication']}")
print(f"Schema type: {schema_type}")
print(f"Cycle:       {base['trial_design']['cycle_length_days']} days")
print(f"AE count:    {len(base['ae_profile'])}")
print(f"Sections:    {list(base.keys())}")

### 4a. Demographics

The demographics section defines the simulated patient population. All values are derived from clinical trial data (CT.gov or PDS), not hardcoded defaults.

**What to look for:**
- **Age**: Should match the typical oncology trial population (~60-65 mean for lung cancer)
- **Sex**: Compare with CT.gov data from Section 1a — the pipeline uses the largest trial's demographics
- **ECOG**: Most patients should be ECOG 0-1; heavy ECOG 2+ suggests a data issue

In [ ]:
demo = base["demographics"]
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

# Age distribution
ax = axes[0]
age = demo["age"]["params"]
samples = np.random.normal(age["mean"], age["std"], 2000)
samples = samples[(samples >= age.get("min", 18)) & (samples <= age.get("max", 95))]
ax.hist(samples, bins=30, color="#3498db", edgecolor="white", alpha=0.8)
ax.axvline(age["mean"], color="red", linestyle="--", linewidth=2, label=f"Mean={age['mean']:.1f}")
ax.set_xlabel("Age (years)")
ax.set_title("Age Distribution")
ax.legend()

# Sex
ax = axes[1]
sex = demo["sex"]["options"]
ax.pie(sex.values(), labels=sex.keys(), autopct="%1.1f%%",
       colors=["#3498db", "#e74c3c"], startangle=90)
ax.set_title("Sex")

# Race
ax = axes[2]
race = demo.get("race", {}).get("options", {})
if race:
    labels = [k[:15] for k in race.keys()]
    ax.barh(labels, list(race.values()), color="#2ecc71")
    ax.set_xlabel("Proportion")
ax.set_title("Race/Ethnicity")

# ECOG
ax = axes[3]
ecog = demo.get("ecog_ps", {}).get("options", {})
if ecog:
    ax.bar([f"ECOG {k}" for k in ecog.keys()], list(ecog.values()), color="#9b59b6")
    ax.set_ylabel("Proportion")
ax.set_title("ECOG PS")

plt.suptitle(f"Demographics — {base['drug_name']}", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### 4b. Adverse Event Profile

The AE profile is the core of the ruleset — it drives the simulation's toxicity model. Each AE has an incidence (0-1), grade distribution, onset timing, and duration.

**What to look for:**
- **High-frequency AEs (>50%)**: Typically hematologic (neutropenia, anemia, thrombocytopenia) for chemo drugs
- **AE count**: Should be 15-30 for most drugs (pipeline caps at 30)
- **Compare with DailyMed chart** from Section 1a: Final frequencies should be lower (0.5x dampening applied)

In [ ]:
aes = sorted(base["ae_profile"], key=lambda x: x["incidence_all_grade"], reverse=True)

# Top 20 AEs by frequency
top_n = 20
top_aes = aes[:top_n]
terms = [ae["ae_term"].replace("_", " ").title()[:25] for ae in top_aes]
freqs = [ae["incidence_all_grade"] * 100 for ae in top_aes]

fig, ax = plt.subplots(figsize=(10, 8))
colors = ["#e74c3c" if f > 50 else "#f39c12" if f > 20 else "#3498db" for f in freqs]
bars = ax.barh(terms[::-1], freqs[::-1], color=colors[::-1])
ax.set_xlabel("Incidence All Grade (%)")
ax.set_title(f"Top {top_n} Adverse Events — {base['drug_name']}")

# Add percentage labels
for bar, val in zip(bars, freqs[::-1]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
            f"{val:.1f}%", va="center", fontsize=9)

plt.tight_layout()
plt.show()

### 4c. Grade Distribution (Top 10 AEs)

Each AE's severity is modeled as a grade distribution (grades 1-5), where proportions sum to ~1.0. The validator auto-corrects fabricated patterns (e.g., identical 50/30/15/5 splits across multiple AEs).

**What to look for:**
- **Hematologic AEs** (neutropenia, anemia): Should have higher grade 3-4 proportions
- **GI AEs** (nausea, vomiting): Should be mostly grade 1-2
- **Varied distributions**: If all AEs look identical, the validator's correction may not have fired — check warnings

In [ ]:
top_n = min(10, len(aes))
top_n_aes = aes[:top_n]

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(top_n)
width = 0.65
grade_colors = {"1": "#2ecc71", "2": "#f1c40f", "3": "#e67e22", "4": "#e74c3c", "5": "#8e44ad"}

bottom = np.zeros(top_n)
for grade in ["1", "2", "3", "4", "5"]:
    vals = [ae["grade_distribution"].get(grade, 0) for ae in top_n_aes]
    if any(v > 0 for v in vals):
        ax.bar(x, vals, width, bottom=bottom, label=f"Grade {grade}",
               color=grade_colors[grade], edgecolor="white", linewidth=0.5)
        bottom += vals

ax.set_xticks(x)
ax.set_xticklabels([ae["ae_term"].replace("_", " ").title()[:18] for ae in top_n_aes],
                    rotation=45, ha="right")
ax.set_ylabel("Proportion")
ax.set_title(f"Severity Grade Distribution — Top {top_n} AEs")
ax.legend(loc="upper right")

plt.tight_layout()
plt.show()

### 4d. Efficacy

Efficacy parameters include response rates (ORR, CR) and survival distributions (PFS, OS). These are extracted from CT.gov trial outcomes and override LLM estimates when available.

**What to look for:**
- **ORR**: For platinum-based SCLC regimens, expect 60-80%; for NSCLC, 30-50%
- **PFS**: Typically 4-6 months for SCLC, 4-8 months for NSCLC
- **OS**: Typically 9-12 months for SCLC, 10-16 months for NSCLC
- If PFS/OS shows "no data", the pipeline lacked CI bounds to compute distribution parameters

In [ ]:
eff = base["efficacy"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# ORR / CR
ax = axes[0]
orr = eff.get("overall_response_rate", 0) or 0
cr = eff.get("complete_response_rate", 0) or 0
ax.bar(["ORR", "CR"], [orr * 100, cr * 100], color=["#3498db", "#2ecc71"], width=0.5)
ax.set_ylabel("%")
ax.set_title("Response Rates")
ax.set_ylim(0, 100)
for i, v in enumerate([orr * 100, cr * 100]):
    ax.text(i, v + 2, f"{v:.1f}%", ha="center", fontweight="bold")

# PFS
ax = axes[1]
pfs = eff.get("progression_free_survival_months", {})
if pfs and pfs.get("params"):
    p = pfs["params"]
    samples = np.random.exponential(p["mean"], 1000)
    samples = samples[(samples >= p.get("min", 0)) & (samples <= p.get("max", 60))]
    ax.hist(samples, bins=30, color="#e67e22", edgecolor="white", alpha=0.8)
    ax.axvline(p["mean"], color="red", linestyle="--", label=f"Median={p['mean']:.1f}m")
    ax.legend()
ax.set_xlabel("Months")
ax.set_title("PFS Distribution")

# OS
ax = axes[2]
os_data = eff.get("overall_survival_months", {})
if os_data and os_data.get("params"):
    p = os_data["params"]
    samples = np.random.exponential(p["mean"], 1000)
    samples = samples[(samples >= p.get("min", 0)) & (samples <= p.get("max", 120))]
    ax.hist(samples, bins=30, color="#9b59b6", edgecolor="white", alpha=0.8)
    ax.axvline(p["mean"], color="red", linestyle="--", label=f"Median={p['mean']:.1f}m")
    ax.legend()
ax.set_xlabel("Months")
ax.set_title("OS Distribution")

plt.suptitle(f"Efficacy — {base['drug_name']}", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### 4e. Administration Schedule

The dosing regimen: which drug, what dose, which route, and on which cycle days. This is one of the strongest-scoring dimensions (~97% accuracy) because doses are extracted deterministically from DailyMed labels and CT.gov descriptions.

In [ ]:
admin = base["administration_schedule"]

print(f"Administration Schedule ({base['trial_design']['cycle_length_days']}-day cycle):")
print("-" * 70)
for entry in admin:
    days_str = ", ".join(str(d) for d in entry.get("cycle_days", []))
    print(f"  {entry['drug_name']:25s}  {entry['dose_per_administration']:20s}  "
          f"{entry['route']:15s}  Days: {days_str}")

### 4f. Comorbidities & Dose Modification Rules

In [ ]:
# Comorbidities
print("Comorbidities:")
for c in base.get("comorbidities", []):
    mods = len(c.get("conditional_modifiers", []))
    print(f"  {c['condition']:30s}  prevalence={c['base_probability']:.1%}  "
          f"modifiers={mods}")

# Dose modification rules
dm_rules = base.get("dose_modification_rules", [])
print(f"\nDose Modification Rules: {len(dm_rules)}")
for rule in dm_rules[:5]:
    ga = rule.get("grade_actions", {})
    actions_str = ", ".join(f"G{g}→{a}" for g, a in sorted(ga.items()))
    print(f"  {rule.get('ae_term', 'N/A'):25s}  {actions_str}")

# AE cascade rules
print(f"\nAE Cascade Rules: {len(base.get('ae_cascade_rules', []))}")
for rule in base.get("ae_cascade_rules", [])[:5]:
    print(f"  {rule.get('trigger_ae', 'N/A'):25s}  -> {rule.get('resulting_ae', 'N/A'):25s}  "
          f"p={rule.get('probability', 0):.0%}")

---
## Summary

This notebook walked through the complete pipeline:

1. **Evidence Collection** — 10 databases (6 API + 3 local + 1 remote) queried in parallel
2. **Stage 1** — 5 parallel LLM calls extract AE freq, severity, onset, triggers, demographics
3. **Stage 2** — Grounding verification (gracefully degrades)
4. **Stage 3** — Final JSON synthesis with aggressive repair
5. **Overrides** — 5 deterministic evidence-based corrections
6. **Re-prompts** — Up to 5 LLM fallback calls for missing data
7. **Validation** — Auto-correction of hallucination patterns
8. **Output** — Split into `base.json` + type overlay

To generate rulesets for other drugs, change `DRUGS` and `INDICATION` in Section 0 and re-run.

---

# Phase B — Clinical Trial Simulation

Uses the rule set generated in Phase A to run a daily patient simulation with hazard functions, Ornstein-Uhlenbeck lab models, and LLM enrichment on event days.

In [ ]:
# ── Simulation Configuration ──
# Uses the same drug/indication from Phase A
DRUG_NAME   = " + ".join(DRUGS)
N_PATIENTS  = 10
TOTAL_DAYS  = 126
MAX_WORKERS = 10
MODEL       = 'gemini-2.0-flash'

RUN_NAME = f'{datetime.now().strftime("%Y%m%d_%H%M%S")}_{DRUG_NAME.replace(" ", "_")}_{N_PATIENTS}pt_{TOTAL_DAYS}d'
RUN_DIR  = ROOT / 'data' / 'runs' / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

print(f'Drug:       {DRUG_NAME}')
print(f'Indication: {INDICATION}')
print(f'Patients:   {N_PATIENTS} × {TOTAL_DAYS} days')
print(f'Run dir:    {RUN_DIR}')

---
## 1. Load Rule Set from Phase A

The rule set was already generated and validated in Phase A above. Now we load it for the simulator.

In [ ]:
# Load the rule set generated in Phase A
output_dir = config.output_dir / OUTPUT_SUBDIR
base_path = output_dir / "base.json"
rule_set_merged = json.loads(base_path.read_text())

# Deep-merge overlay (route-specific fields like infusion_duration_minutes)
overlay_files = [f for f in output_dir.iterdir() if f.name != "base.json" and f.suffix == ".json" and "agent_log" not in f.name]
if overlay_files:
    overlay = json.loads(overlay_files[0].read_text())
    for key, val in overlay.items():
        if key == "administration_schedule" and key in rule_set_merged:
            base_admin = {a["drug_name"]: a for a in rule_set_merged[key]}
            for entry in val:
                drug = entry["drug_name"]
                if drug in base_admin:
                    base_admin[drug].update(entry)
                else:
                    base_admin[drug] = entry
            rule_set_merged[key] = list(base_admin.values())
        else:
            rule_set_merged[key] = val

# Save to RUN_DIR for the orchestrator
rule_set_path = RUN_DIR / 'rule_set.json'
rule_set_path.write_text(json.dumps(rule_set_merged, indent=2, ensure_ascii=False))

ae_profile = rule_set_merged.get('ae_profile', [])

print(f'Rule set loaded: {len(ae_profile)} AEs')
print(f'Drug: {rule_set_merged.get("drug_name", DRUG_NAME)}')
print(f'Saved to: {rule_set_path}')


In [ ]:
# Detect schema type
schema_type = "unknown"
if overlay_files:
    schema_type = overlay_files[0].stem
print(f"Schema type: {schema_type}")
print(f"Rule set keys: {list(rule_set_merged.keys())}")

In [ ]:
# Inspect the rule set that will drive the simulation
rs = rule_set_merged
print(f'Drug: {rs.get("drug_name", DRUG_NAME)}')
print(f'Indication: {rs.get("indication", INDICATION)}')

cycle = rs.get("trial_design", {})
print(f'Cycle: {cycle.get("cycle_length_days", 21)} days')

eff = rs.get("efficacy", {})
orr = eff.get("overall_response_rate", 0)
print(f'Expected ORR: {orr*100:.1f}%' if orr < 1 else f'Expected ORR: {orr}%')

admin = rs.get("administration_schedule", [])
for a in admin:
    print(f'  {a.get("drug_name")}: {a.get("dose_per_administration")} {a.get("route", "")} Day {a.get("cycle_days", [])}')

mort = rs.get("mortality_model", {})
ch = mort.get("channels", {})
print(f'\nMortality: baseline={mort.get("baseline_annual_mortality")}, '
      f'dp={bool(ch.get("disease_progression"))}, tt={bool(ch.get("treatment_toxicity"))}')

print(f'Dose mod rules: {len(rs.get("dose_modification_rules", []))}')
print(f'Cascade rules: {len(rs.get("ae_cascade_rules", []))}')

print(f'\n── AE Profile (top 15 by incidence) ──')
ae_sorted = sorted(ae_profile, key=lambda x: x.get('incidence_all_grade', 0), reverse=True)
ae_rows = []
for ae in ae_sorted[:15]:
    name = ae.get('ae_term', ae.get('name', ae.get('ae_name', '?')))
    inc = ae.get('incidence_all_grade', 0)
    ae_rows.append({'AE': name.replace('_', ' ').title(), 'Incidence %': f'{inc*100:.1f}'})

if ae_rows:
    display(pd.DataFrame(ae_rows))

---
## 2. Rule Set Validation

Two levels of validation:
1. **Schema validation** — structural correctness
2. **Clinical plausibility** — hallucination detection (severity fabrication, onset plausibility, FAERS coverage)

In [ ]:
from rule_engine.validator import validate_rule_set
from rule_engine.schema import RuleSet

try:
    rule_set_obj_for_val = RuleSet.model_validate(json.loads(
        (output_dir / "base.json").read_text()
    ))
    warnings_list = validate_rule_set(rule_set_obj_for_val, bundle=bundle)
    if warnings_list:
        print(f'⚠ {len(warnings_list)} warnings:')
        for w in warnings_list[:10]:
            print(f'  - {w}')
    else:
        print('✅ Rule set passed all validation checks')
except Exception as e:
    warnings_list = []
    print(f'Validation skipped: {e}')

---
## 3. Phase 1 — Patient Generation

Each patient is generated through a 4-step process:
1. **Demographics** — age, sex, race sampled from rule set distributions
2. **Comorbidities** — LLM adjusts base probabilities given demographics
3. **Baseline values** — LLM generates internally consistent labs/vitals (e.g., CKD → creatinine↑)
4. **Persona** — LLM assigns personality type that determines symptom reporting behavior

In [ ]:
from src.orchestrator_v2 import SimulationRunnerV2

runner = SimulationRunnerV2(
    drug_name=DRUG_NAME,
    indication=INDICATION,
    model=MODEL,
    data_dir=str(RUN_DIR),
    seed=42,
)
runner.rule_set = rule_set_merged

print(f'Generating {N_PATIENTS} patients (parallel, {MAX_WORKERS} workers)...')
t0 = time.time()
patients = runner.create_patients_parallel(n=N_PATIENTS, max_workers=MAX_WORKERS)
print(f'\n✅ {len(patients)} patients generated in {time.time()-t0:.1f}s')

In [ ]:
# Patient overview
pt_rows = []
for p in patients:
    dm = p.get('emr', {}).get('demographics', {})
    dx = p.get('emr', {}).get('diagnosis', {})
    persona = p.get('persona', {})
    pt_rows.append({
        'ID': p.get('patient_id'),
        'Age': dm.get('age'),
        'Sex': dm.get('sex'),
        'Race': dm.get('race', '')[:15],
        'ECOG': p.get('emr', {}).get('baseline_ecog'),
        'Stage': dx.get('stage', ''),
        'Persona': persona.get('type', ''),
    })
display(pd.DataFrame(pt_rows))

# Inspect one patient's full profile
print(f'\n── Sample patient ({patients[0]["patient_id"]}) baseline labs ──')
labs = patients[0].get('emr', {}).get('baseline_labs', {})
for k, v in list(labs.items())[:10]:
    print(f'  {k}: {v}')

---
## 4. Phase 2 — Daily Simulation

The daily simulation engine uses **code-based probabilistic models** (not LLM for every day):

| Component | Method | Purpose |
|-----------|--------|---------|
| AE onset/resolution | Hazard functions | When AEs start/stop |
| AE grade transitions | Markov probabilities | Grade worsening/improving |
| Labs & vitals | Ornstein-Uhlenbeck process | Physiological continuity |
| AE → Lab shifts | 3-layer causal model | hepatitis → ALT↑, neutropenia → ANC↓ |
| Tumor response | Sigmoid model | Tumor shrinkage/growth over time |
| LLM enrichment | Gemini Flash | Event-day narrative & clinical detail |

**LLM is only called on "event days"** (AE onset, grade change, treatment modification) — quiet days are pure code.

In [ ]:
print(f'Starting simulation: {N_PATIENTS} patients × {TOTAL_DAYS} days (natural mode)')
print(f'Workers: {MAX_WORKERS}\n')

t0 = time.time()
all_results = runner.run_parallel(
    patients, total_days=TOTAL_DAYS, mode='natural', max_workers=MAX_WORKERS, save=True
)
sim_elapsed = time.time() - t0

print(f'\n✅ Simulation complete in {sim_elapsed:.1f}s ({sim_elapsed/N_PATIENTS:.1f}s/patient)')
print(f'Total days simulated: {sum(len(v) for v in all_results.values())}')

---
## 5. Simulation Output Exploration

Let's inspect what the simulation produced — day-by-day patient data including labs, vitals, AEs, and tumor response.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.facecolor'] = '#0d1117'
matplotlib.rcParams['axes.facecolor'] = '#161b22'
matplotlib.rcParams['text.color'] = '#e6edf3'
matplotlib.rcParams['axes.labelcolor'] = '#8b949e'
matplotlib.rcParams['xtick.color'] = '#8b949e'
matplotlib.rcParams['ytick.color'] = '#8b949e'

# Load simulation data for first patient
pid = patients[0]['patient_id']
sim_file = RUN_DIR / 'simulations' / f'{pid}_natural.jsonl'
days_data = []
with open(sim_file) as f:
    for line in f:
        days_data.append(json.loads(line))

print(f'Patient {pid}: {len(days_data)} days of data')
print(f'\n── Day 1 snapshot ──')
d1 = days_data[0] if days_data else {}
obj = d1.get('objective', {})
print(f'  Location: {obj.get("location")}')
print(f'  Treatment: {obj.get("treatment_status")}')
print(f'  ECOG: {obj.get("ecog")}')
print(f'  Active AEs: {len(obj.get("active_aes", []))}')

# Find a day with AEs
ae_days = [d for d in days_data if d.get('objective', {}).get('active_aes')]
if ae_days:
    sample = ae_days[min(3, len(ae_days)-1)]
    print(f'\n── Day {sample["day"]} (with AEs) ──')
    for ae in sample['objective']['active_aes']:
        print(f'  {ae.get("ae", "?")}: Grade {ae.get("grade")}, onset Day {ae.get("onset_day")}')

In [ ]:
# Lab time series — demonstrates Ornstein-Uhlenbeck continuity + AE-Lab causal model
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle(f'Lab Time Series — {pid}', fontsize=14, color='#58a6ff')

lab_keys = [('ANC', 'ANC (×10⁹/L)'), ('creatinine', 'Creatinine (mg/dL)'),
            ('ALT', 'ALT (U/L)'), ('hemoglobin', 'Hemoglobin (g/dL)')]

for ax, (key, label) in zip(axes.flat, lab_keys):
    day_nums, values = [], []
    for d in days_data:
        labs = d.get('objective', {}).get('labs', {}) or d.get('LB', {})
        if not labs:
            continue
        val = None
        for lk, lv in labs.items():
            if key.lower() in lk.lower():
                val = lv.get('value', lv) if isinstance(lv, dict) else lv
                break
        if val is not None:
            try:
                day_nums.append(d['day'])
                values.append(float(val))
            except (ValueError, TypeError):
                pass
    if values:
        ax.plot(day_nums, values, color='#39d2c0', linewidth=1.2, alpha=0.9)
        ax.fill_between(day_nums, values, alpha=0.1, color='#39d2c0')
    ax.set_title(label, fontsize=11, color='#e6edf3')
    ax.set_xlabel('Day')
    ax.grid(alpha=0.15)

plt.tight_layout()
plt.show()

In [ ]:
# AE Timeline Heatmap — all patients
all_ae_events = []
for pid_key, results in all_results.items():
    for d in results:
        for ae in d.get('objective', {}).get('active_aes', []):
            all_ae_events.append({
                'patient': pid_key, 'day': d['day'],
                'ae': ae.get('ae', '?'), 'grade': ae.get('grade', 1)
            })

if all_ae_events:
    ae_df = pd.DataFrame(all_ae_events)
    ae_counts = ae_df.groupby('ae').agg(patients=('patient', 'nunique'), events=('day', 'count')).sort_values('patients', ascending=False)
    print(f'Total AE events: {len(ae_df)}, unique terms: {ae_df["ae"].nunique()}')
    display(ae_counts.head(15))
else:
    print('No AEs recorded (possible with very short simulations)')

In [ ]:
# Waterfall Plot — Best Tumor Response
waterfall = []
for pid_key, results in all_results.items():
    best = None
    for d in results:
        tumor = d.get('objective', {}).get('tumor') or {}
        change = tumor.get('estimated_change_pct')
        if change is not None:
            if best is None or change < best:
                best = change
    if best is not None:
        waterfall.append({'patient': pid_key, 'change': best})

if waterfall:
    wf = sorted(waterfall, key=lambda x: x['change'])
    colors = ['#3fb950' if w['change'] <= -30 else '#d29922' if w['change'] <= 0 else '#f85149' for w in wf]
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.bar(range(len(wf)), [w['change'] for w in wf], color=colors, width=0.7)
    ax.axhline(-30, color='#58a6ff', linestyle='--', alpha=0.5, label='PR threshold (-30%)')
    ax.axhline(20, color='#f85149', linestyle='--', alpha=0.5, label='PD threshold (+20%)')
    ax.set_ylabel('Best Change from Baseline (%)')
    ax.set_title('Waterfall Plot — Best Tumor Response', color='#58a6ff')
    ax.set_xticks(range(len(wf)))
    ax.set_xticklabels([w['patient'] for w in wf], rotation=45, fontsize=8)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.15, axis='y')
    plt.tight_layout()
    plt.show()
else:
    print('No tumor data available')

---
## 6. CRF Mapping — CDASH Conversion

Raw simulation data is mapped to **CDASH-compliant CRF domains**: AE, LB, VS, EC, CM, DS, RS, TU, DD, PE, EG.

In [ ]:
from src.crf_mapper import map_day_record, map_patient_record

# Map a sample day record
sample_day = ae_days[0] if ae_days else days_data[5] if len(days_data) > 5 else days_data[0]
mapped = map_day_record(sample_day, patients[0])

print(f'Day {sample_day["day"]} → CDASH domains: {list(mapped.keys())}\n')

# Show AE domain
if mapped.get('AE'):
    print('── AE Domain ──')
    display(pd.DataFrame(mapped['AE'][:5]))

# Show LB domain (labs)
if mapped.get('LB'):
    print('\n── LB Domain (Labs) ──')
    lb_rows = [{'Test': r.get('LBTEST',''), 'Result': r.get('LBORRES',''), 'Unit': r.get('LBORRESU',''), 'Baseline': r.get('LBBLFL','')} for r in mapped['LB'][:8]]
    display(pd.DataFrame(lb_rows))

---
## 7. Validation — Simulation vs Rule Set

Statistical comparison of simulated data against the generated rule set using **SMD, TOST equivalence tests, and chi-square tests**.

In [ ]:
from validation.extract_sim_stats import extract_all_stats

sim_stats = extract_all_stats(str(RUN_DIR), mode='natural')

print('── Extracted Statistics ──')
print(f'Demographics: {list(sim_stats.get("demographics", {}).keys())}')
print(f'AE terms: {len(sim_stats.get("ae_statistics", {}).get("ae_by_term", {}))}')
print(f'Efficacy ORR: {sim_stats.get("efficacy", {}).get("orr_pct", "?")}%')

In [ ]:
# Compare simulated AE rates vs rule set expectations
sim_ae = sim_stats.get('ae_statistics', {}).get('ae_by_term', {})
rule_ae = {}
for ae in ae_profile:
    name = ae.get('ae_term', ae.get('name', '?')).lower().replace(' ', '_')
    rule_ae[name] = ae.get('incidence_all_grade', 0) * 100

comparison_rows = []
for ae_name, rule_freq in sorted(rule_ae.items(), key=lambda x: -x[1])[:15]:
    sim_freq = 0
    for sk, sv in sim_ae.items():
        if ae_name.replace('_', '') in sk.lower().replace('_', '').replace(' ', ''):
            sim_freq = sv.get('all_grade_pct', 0)
            break
    comparison_rows.append({
        'AE': ae_name.replace('_', ' ').title(),
        'Rule Set (%)': round(rule_freq, 1),
        'Simulated (%)': round(sim_freq, 1),
        'Delta': round(sim_freq - rule_freq, 1),
    })

print('── AE Frequency: Rule Set vs Simulation ──')
display(pd.DataFrame(comparison_rows))

---
## 8. CSR-Level Statistics (ICH E3)

Comprehensive Clinical Study Report statistics — the same tables displayed on the web frontend's Statistical Analysis page.

In [ ]:
from validation.csr_stats import compute_csr_stats

csr = compute_csr_stats(str(RUN_DIR), mode='natural')

# Disposition
disp = csr['disposition']
print(f'── Disposition (N={disp["enrolled"]}) ──')
print(f'  Completed: {disp["completed"]["n"]} ({disp["completed"]["pct"]}%)')
print(f'  Discontinued: {disp["discontinued"]["n"]} ({disp["discontinued"]["pct"]}%)')
print(f'  Deaths: {disp["deaths"]["n"]} ({disp["deaths"]["pct"]}%)')

# Efficacy
eff = csr['efficacy']
print(f'\n── Efficacy ──')
print(f'  ORR: {eff["orr"]["pct"]}% (CR={eff["best_response"].get("CR",{}).get("n",0)}, PR={eff["best_response"].get("PR",{}).get("n",0)})')
print(f'  DCR: {eff["dcr"]["pct"]}%')
print(f'  Median OS: {eff["km_os"].get("median", "NR")} days')
print(f'  Median PFS: {eff["km_pfs"].get("median", "NR")} days')

# Safety summary
safe = csr['safety']['summary']
N = csr['safety']['n']
print(f'\n── Safety (N={N}) ──')
print(f'  Any AE: {safe["any_ae"]["n"]}/{N} ({safe["any_ae"]["pct"]}%)')
print(f'  Grade ≥3: {safe["grade_gte3"]["n"]}/{N} ({safe["grade_gte3"]["pct"]}%)')
print(f'  SAE: {safe["sae"]["n"]}/{N} ({safe["sae"]["pct"]}%)')
print(f'  Fatal: {safe["fatal"]["n"]}/{N} ({safe["fatal"]["pct"]}%)')

In [ ]:
# AE by Preferred Term table (CSR Table 14.3.1)
ae_table = csr['safety']['by_term'][:15]
ae_rows = []
for t in ae_table:
    gd = t.get('grade_dist', {})
    ae_rows.append({
        'Preferred Term': t['term'],
        'All Grades n (%)': f"{t['all_grade']['n']} ({t['all_grade']['pct']}%)",
        'Grade ≥3 n (%)': f"{t['grade_gte3']['n']} ({t['grade_gte3']['pct']}%)",
        'G1': gd.get('1', 0), 'G2': gd.get('2', 0),
        'G3': gd.get('3', 0), 'G4': gd.get('4', 0),
        'Median Onset': f"Day {t['onset_median']}" if t.get('onset_median') else '—',
    })

print('── AE by Preferred Term (Top 15) ──')
display(pd.DataFrame(ae_rows))

In [ ]:
# Treatment Exposure
tx = csr['treatment']
print('── Treatment Exposure ──')
print(f'  Duration: {tx["duration"]["mean"]} ± {tx["duration"]["std"]} days (median {tx["duration"]["median"]})')
print(f'  Cycles: {tx["cycles"]["mean"]} ± {tx["cycles"]["std"]} (median {tx["cycles"]["median"]})')
print(f'  Dose reduction: {tx["dose_reduction_all"]["n"]} ({tx["dose_reduction_all"]["pct"]}%)')
print(f'  Dose interruption: {tx["dose_interruption_all"]["n"]} ({tx["dose_interruption_all"]["pct"]}%)')
print(f'  Discontinuation: {tx["discontinuation_all"]["n"]} ({tx["discontinuation_all"]["pct"]}%)')

# Demographics
demo = csr['demographics']
print(f'\n── Demographics ──')
print(f'  Age: {demo["age"]["mean"]} ± {demo["age"]["std"]} (range {demo["age"]["min"]}–{demo["age"]["max"]})')
for sex, info in demo.get('sex', {}).items():
    print(f'  {sex}: {info["n"]} ({info["pct"]}%)')

---
## 9. Summary

### Pipeline Recap

In [ ]:
total_ae_events = sum(
    sum(1 for d in results for ae in d.get('objective', {}).get('active_aes', []))
    for results in all_results.values()
)
total_days_sim = sum(len(v) for v in all_results.values())

summary = f"""
# Full Pipeline Summary

| Phase | Detail | Time |
|-------|--------|------|
| Drug | **{DRUG_NAME}** ({INDICATION}) | — |
| Rule Set | {len(ae_profile)} AEs, {schema_type} schema | ~2-3 min |
| Validation | {len(warnings_list)} warnings | <1s |
| Patients | **{len(patients)}** generated | ~1-2 min |
| Simulation | **{total_days_sim}** patient-days | {sim_elapsed:.0f}s |
| Total AE events | {total_ae_events} | — |
| ORR | {eff['orr']['pct']}% | — |
| Median OS | {eff['km_os'].get('median', 'NR')} days | — |
| Any AE | {safe['any_ae']['pct']}% | — |
| Grade ≥3 | {safe['grade_gte3']['pct']}% | — |
| LLM | Gemini 2.0 Flash | — |
| Output | `{RUN_DIR.name}` | — |

**Zero hardcoding.** Change `DRUG_NAME` and `INDICATION` at the top to simulate any drug.
"""
display(Markdown(summary))